In [4]:
# main_train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results" 

# 12 类
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.0005
NUM_EPOCHS = 50

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# ================= 1. 数据集定义 =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = [] 
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), 
            transforms.ToTensor()
        ])
        
        if not os.path.exists(root_dir):
            print(f"Path not found: {root_dir}")
            return
            
        print(f"Loading {root_dir} into RAM...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.png")])
        
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    snr = -999.0
                    if has_snr:
                        try:
                            fname = p.stem 
                            snr_str = fname.split('_')[-1]
                            snr = float(snr_str)
                        except: pass
                    
                    with Image.open(str(p)).convert("RGB") as img:
                        tensor = self.transform(img)
                        self.data.append(tensor[:2, :, :]) 
                        self.labels.append(label)
                        self.snrs.append(snr)
                except Exception as e:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): 
        return self.data[idx], self.labels[idx], self.snrs[idx]

# ================= 2. 模型定义 =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 辅助绘图函数 =================
def plot_training_curves(history, save_path):
    epochs = range(1, len(history['train_acc']) + 1)
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss') 
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10)) # 加大图片以容纳12个类别
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ================= 4. 主训练逻辑 =================
def main():
    print("Initializing Datasets...")
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False) 
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True) 
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    if len(train_ds) == 0:
        print("No data found! Run data_generator.py first.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    best_val_acc = 0.0
    
    print(f"\nStarting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels, _ in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        # Validation
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                val_running_loss += loss.item() * imgs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        
        log_msg = (f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                   f"Loss: {epoch_loss:.4f}/{val_loss:.4f} | "
                   f"Acc: {epoch_acc:.4f}/{val_acc:.4f}")
        if is_best: log_msg += "  [New Best!]"
        print(log_msg)

    plot_training_curves(history, os.path.join(RESULTS_DIR, "training_curves.png"))
    
    # --- 5. 加载最佳模型 ---
    print("\nLoading Best Model for Evaluation...")
    if os.path.exists(BEST_MODEL_PATH):
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    # --- Per-SNR Evaluation ---
    print("Generating Per-SNR Confusion Matrices...")
    val_results = {'true': [], 'pred': [], 'snr': []}
    
    with torch.no_grad():
        for imgs, labels, snrs in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            val_results['true'].extend(labels.cpu().numpy())
            val_results['pred'].extend(preds.cpu().numpy())
            val_results['snr'].extend(snrs)
            
    y_true = np.array(val_results['true'])
    y_pred = np.array(val_results['pred'])
    snrs = np.array(val_results['snr'])
    
    snr_bins = [(-5, 0), (0, 5), (5, 10), (10, 15), (15, 20), (20, 30)]
    
    for (low, high) in snr_bins:
        mask = (snrs >= low) & (snrs < high)
        if np.sum(mask) == 0: continue
        
        subset_true = y_true[mask]
        subset_pred = y_pred[mask]
        subset_acc = np.mean(subset_true == subset_pred)
        
        title = f"SNR {low}-{high}dB (Acc: {subset_acc:.2%})"
        filename = os.path.join(RESULTS_DIR, f"cm_snr_{low}_{high}.png")
        plot_confusion_matrix(subset_true, subset_pred, title, filename)
        print(f"  -> Saved {filename}")

    # --- 6. 真实数据测试 ---
    if len(real_ds) > 0:
        print("\nRunning Real Data Test...")
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=False)
        real_true = []
        real_pred = []
        
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                real_true.extend(labels.cpu().numpy())
                real_pred.extend(preds.cpu().numpy())
        
        real_acc = np.mean(np.array(real_true) == np.array(real_pred))
        print(f"Real Data Accuracy: {real_acc:.4f}")
        plot_confusion_matrix(real_true, real_pred, 
                             f"Real Data Confusion Matrix (Acc: {real_acc:.2%})",
                             os.path.join(RESULTS_DIR, "cm_real_data.png"))
        print(f"Final Results saved to {RESULTS_DIR}")

if __name__ == "__main__":
    main()

Initializing Datasets...
Loading ./dataset_dual_RA\sim_train into RAM...


100%|██████████| 7632/7632 [00:02<00:00, 3288.60img/s]


Loading ./dataset_dual_RA\sim_val into RAM...


100%|██████████| 1968/1968 [00:00<00:00, 3597.24img/s]


Loading ./dataset_dual_RA\real_test into RAM...


100%|██████████| 43938/43938 [00:11<00:00, 3793.68img/s]



Starting Training for 50 epochs...
Epoch 1/50 | Loss: 1.9997/1.4236 | Acc: 0.2858/0.5137  [New Best!]
Epoch 2/50 | Loss: 1.3511/0.9231 | Acc: 0.5088/0.7027  [New Best!]
Epoch 3/50 | Loss: 1.0397/0.7449 | Acc: 0.6186/0.7454  [New Best!]
Epoch 4/50 | Loss: 0.8957/0.6807 | Acc: 0.6736/0.7449
Epoch 5/50 | Loss: 0.8191/0.6489 | Acc: 0.6996/0.7546  [New Best!]
Epoch 6/50 | Loss: 0.7740/0.6159 | Acc: 0.7113/0.7754  [New Best!]
Epoch 7/50 | Loss: 0.7297/0.5985 | Acc: 0.7262/0.7637
Epoch 8/50 | Loss: 0.7065/0.5955 | Acc: 0.7313/0.7688
Epoch 9/50 | Loss: 0.6993/0.5671 | Acc: 0.7334/0.7820  [New Best!]
Epoch 10/50 | Loss: 0.6797/0.5597 | Acc: 0.7372/0.7815
Epoch 11/50 | Loss: 0.6638/0.5394 | Acc: 0.7454/0.7846  [New Best!]
Epoch 12/50 | Loss: 0.6446/0.5392 | Acc: 0.7513/0.7881  [New Best!]
Epoch 13/50 | Loss: 0.6454/0.5423 | Acc: 0.7453/0.7861
Epoch 14/50 | Loss: 0.6272/0.5380 | Acc: 0.7589/0.7815
Epoch 15/50 | Loss: 0.6200/0.5276 | Acc: 0.7600/0.7881
Epoch 16/50 | Loss: 0.6121/0.5206 | Acc: 0.7